# Setup

Import librarie and configure the GPU.

In [2]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.transforms import functional as F
from PIL import Image
import numpy as np
import os
import glob
from tqdm.notebook import tqdm
import torch.cuda.amp as amp

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device: {device}")

Using device: cuda


# Dataset

YoloV8 and FasterRCNN expect different labels, so data manipulation must be performed.

YoloV8: class x_center y_center width height
FasterRCNN: tensor image + target (boxes [x_min, y_min, x_max, y_max] and class)
    Ex: loss_dict = model(images, targets) -> this is what the model expects

In [7]:
class YoloCupDataset(Dataset):
    def __init__(self, img_dir, label_dir, transforms=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transforms = transforms
        self.img_files = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
        
    def __getitem__(self, index):
        img_path = self.img_files[index]
        img = Image.open(img_path).convert("RGB")
        w_img, h_img = img.size
        
        filename = os.path.basename(img_path)
        label_path = os.path.join(self.label_dir, filename.replace(".jpg", ".txt"))
        
        boxes = []
        labels = []
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()
                
            for line in lines:
                parts = line.strip().split()
                x_c, y_c, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])

                # convert to pixels
                x_c, y_c, w, h = x_c * w_img, y_c * h_img, w * w_img, h * h_img

                # convert center to corner coordinates
                x_min = x_c - (w / 2)
                y_min = y_c - (h / 2)
                x_max = x_c + (w / 2)
                y_max = y_c + (h / 2)
                
                boxes.append([x_min, y_min, x_max, y_max])
                labels.append(1)

        # due to negative samples, treat cases with no cup
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        # convert the lists to tensors
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        
        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([index])}
        
        if self.transforms is not None:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.img_files)

def get_transform():
    return torchvision.transforms.Compose([torchvision.transforms.ToTensor()])

val_img_dir = '../mug_coco_yolo/images/val2017'
val_label_dir = '../mug_coco_yolo/labels/val2017'
train_img_dir = '../mug_coco_yolo/images/train2017'
train_label_dir = '../mug_coco_yolo/labels/train2017'

full_val_dataset = YoloCupDataset(val_img_dir, val_label_dir, get_transform())
full_train_dataset = YoloCupDataset(train_img_dir, train_label_dir, get_transform())

# take a subset for training due to limit hardware resources
# validation is identical with test, so it's actually just testing 
val_subset = Subset(full_val_dataset, torch.arange(300))
train_subset = Subset(full_train_dataset, torch.arange(200))

print(f"Validation Subset: {len(val_subset)} images")
print(f"Training Subset:   {len(train_subset)} images")

Validation Subset: 300 images
Training Subset:   200 images


# Evluation

Function for model evaluation using mAP@0.50 and mAP@0.50:0.95 metrics.
Drawing the boxes over detected objects is also manually performed.

In [4]:
import torchvision.transforms.functional as F
from PIL import ImageDraw

def draw_predictions(
    image,
    boxes,
    scores,
    score_threshold=0.5,
    color="red"
):
    draw = ImageDraw.Draw(image)

    for box, score in zip(boxes, scores):
        if score < score_threshold:
            continue

        x1, y1, x2, y2 = box.tolist()
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        draw.text((x1, y1), f"{score:.2f}", fill=color)

    return image

In [5]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

def collate_fn(batch):
    return tuple(zip(*batch))

def evaluate_model(
    model,
    dataset,
    device,
    score_threshold=0.05,
    pred_class_id=1,
):
    # replace default collate_fn function with the one above 
    # to ensure item separation
    data_loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_fn)

    # computed over all images and sorted by confidence
    metric = MeanAveragePrecision(iou_type="bbox")
    # switch off training mode (deterministic evaluation)
    model.eval()

    # no gradients in evaluation (test/validation) -> saves memory & faster
    with torch.no_grad():
        for idx, (images, targets) in enumerate(tqdm(data_loader, desc="Evaluating (mAP)")):
            # the model expects a list of images
            images = [img.to(device) for img in images]
            outputs = model(images)

            # batch_size = 1 => one image, prediction, target
            pred = outputs[0]
            tgt = targets[0]

            boxes = pred["boxes"]
            scores = pred["scores"]
            labels = pred["labels"]

            keep = (labels == pred_class_id) & (scores >= score_threshold)
            num_kept = int(keep.sum().item())

            preds = [{
                "boxes": boxes[keep].detach().cpu(),
                "scores": scores[keep].detach().cpu(),
                "labels": torch.ones((num_kept,), dtype=torch.int64),
            }]

            num_gt = int(tgt["boxes"].shape[0])
            target = [{
                "boxes": tgt["boxes"].detach().cpu(),
                "labels": torch.ones((num_gt,), dtype=torch.int64),
            }]

            metric.update(preds, target)

            # draw box over predicted object
            img_pil = F.to_pil_image(images[0].detach().cpu())
            img_pil = draw_predictions(
                img_pil,
                boxes[keep].detach().cpu(),
                scores[keep].detach().cpu(),
                score_threshold=score_threshold
            )
            save_path = f"../outputs/val_{idx:03d}.jpg"
            img_pil.save(save_path)

    return metric.compute()

# Baseline Model

Evaluate the baseline FasterRCNN model using the validation set.

In [10]:
print("--- LOADING BASELINE MODEL ---")
baseline_model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
baseline_model.to(device)

print("--- RUNNING BASELINE EVALUATION ---")
b_map = evaluate_model(
    baseline_model,
    val_subset,
    device,
    pred_class_id=47
)

print("Baseline:")
print("mAP@50     :", b_map["map_50"].item())
print("mAP@50-95  :", b_map["map"].item())

--- LOADING BASELINE MODEL ---
--- RUNNING BASELINE EVALUATION ---


Evaluating (mAP):   0%|          | 0/300 [00:00<?, ?it/s]

Baseline:
mAP@50     : 0.6824472546577454
mAP@50-95  : 0.4704633057117462


# Training

We initialize the Faster R-CNN model and adapt it to our single class cup detection.   
Training is performed using stochastic (update weights after each training sample) gradient descent on a subset of the filtered COCO dataset.

In [7]:
model = fasterrcnn_resnet50_fpn(weights="DEFAULT")

# prevent weights from updating to ensure faster training (since the dataset is small)
for param in model.backbone.parameters():
    param.requires_grad = False

# replace classifier's head
in_features = model.roi_heads.box_predictor.cls_score.in_features
# make the head 2 class -> cup & background
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
# all inputs on same device (GPU)
model.to(device)

train_loader = DataLoader(train_subset, batch_size=2, shuffle=True, num_workers=0, collate_fn=collate_fn)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

NUM_EPOCHS = 5
print(f"--- STARTING FINE-TUNING ({NUM_EPOCHS} Epochs) ---")

# train the head on the subset 
for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    prog_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}", unit="batch")
    
    for images, targets in prog_bar:
        # images and targets must be on the same device as model (GPU)
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # learning step -> compute gradients for trainable params
        # adjust the parameters to reduce the loss

        # clears old gradients 
        optimizer.zero_grad()
        losses.backward()
        # update params
        optimizer.step()
        
        epoch_loss += losses.item()
        prog_bar.set_postfix(loss=f"{losses.item():.4f}")
        
    print(f"Epoch {epoch+1} Loss: {epoch_loss/len(train_loader):.4f}")

print("--- TRAINING COMPLETE ---")

--- STARTING FINE-TUNING (5 Epochs) ---


Epoch 1:   0%|          | 0/100 [00:00<?, ?batch/s]

Epoch 1 Loss: 0.1321


Epoch 2:   0%|          | 0/100 [00:00<?, ?batch/s]

Epoch 2 Loss: 0.0756


...

Epoch 3 Loss: 0.0693


Epoch 4:   0%|          | 0/100 [00:00<?, ?batch/s]

Epoch 4 Loss: 0.0637


Epoch 5:   0%|          | 0/100 [00:00<?, ?batch/s]

Epoch 5 Loss: 0.0599
--- TRAINING COMPLETE ---


# Fine-tuned model

Evaluate the fine-tuned FasterRCNN model using the validation set.

In [28]:
print("--- RUNNING FINE-TUNED EVALUATION ---")
ft_map = evaluate_model(
    model,
    val_subset,
    device,
    pred_class_id=1
)

print("Fine-tuned:")
print("mAP@50     :", ft_map["map_50"].item())
print("mAP@50-95  :", ft_map["map"].item())

--- RUNNING FINE-TUNED EVALUATION ---


Evaluating (mAP):   0%|          | 0/780 [00:00<?, ?it/s]

Fine-tuned:
mAP@50     : 0.6540079712867737
mAP@50-95  : 0.42467403411865234
